# Polymarket wallet skill — Colab runner

**You probably don't need this notebook.** The same scan runs automatically via
GitHub Actions (`.github/workflows/polymarket-scan.yml`) and commits its report
to `results/polymarket/REPORT.md`. Use this only to iterate interactively.

This notebook is a thin wrapper around `scripts/polymarket_scan.py` — the *same*
code path the Action runs, and the one covered by tests. Earlier versions of this
notebook duplicated the pipeline inline, which meant notebook-only bugs. One
tested path is better than two.

In [ ]:
!git clone -q https://github.com/parsiqman/flow-signal.git 2>/dev/null || true
%cd /content/flow-signal
!git pull -q
!pip install -q pandas numpy matplotlib
print('ready')

## 1. Self-test first

Runs the whole analysis on synthetic data with known ground truth. If this
fails, the problem is our code. If this passes and the live scan fails, the
problem is the API. That distinction is the entire point of running it first.

In [ ]:
!python tests/test_polymarket.py
!python scripts/polymarket_scan.py --offline --out /tmp/selftest 2>&1 | tail -5

## 2. Live scan

Start at 200 markets to confirm it works end to end, then raise it. Responses
are cached to `/tmp/pm_cache`, so re-running is cheap and reproducible.

If this fails with a **field-resolution error**, the API shape has drifted —
the message lists the actual response columns. Add the correct spelling to
`TRADE_FIELD_CANDIDATES` / `MARKET_FIELD_CANDIDATES` in
`src/polymarket/client.py`.

In [ ]:
!python scripts/polymarket_scan.py \
    --markets 200 \
    --min-volume 5000 \
    --min-trades 20 \
    --cache /tmp/pm_cache \
    --out /content/pm_results

## 3. Read the verdict

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
p = Path('/content/pm_results/REPORT.md')
display(Markdown(p.read_text() if p.exists() else '**No report — check the log above.**'))

In [ ]:
import pandas as pd, json
from pathlib import Path

f = Path('/content/pm_results/wallet_scores.csv')
if f.exists():
    scores = pd.read_csv(f)
    print(f'{len(scores):,} scored wallets\n')
    print(scores.head(20)[['wallet','n_trades','n_eff','edge_per_share',
                           'roi','t_stat','clears_luck']].to_string(index=False))
else:
    print('no wallet_scores.csv — the scan did not get that far')

## 4. Look at the distribution

The shape matters more than the maximum. A population of pure noise still
produces a right tail that looks like talent — the red line is where luck alone
would put the best of this many wallets.

In [ ]:
import matplotlib.pyplot as plt
if f.exists() and len(scores):
    res = json.loads(Path('/content/pm_results/scan_result.json').read_text())
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].hist(scores['edge_per_share'] * 100, bins=60)
    ax[0].set_xlabel('edge (cents/share)'); ax[0].set_ylabel('wallets')
    ax[0].set_title('Edge distribution'); ax[0].grid(alpha=.3)
    ax[1].hist(scores['t_stat'].dropna(), bins=60)
    ax[1].axvline(res.get('t_needed', 4), color='r', ls='--',
                  label=f"luck bar t={res.get('t_needed')}")
    ax[1].set_xlabel('t-statistic'); ax[1].set_title('Skill vs luck')
    ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout()

## 5. What the answer means

| Verdict | What to do |
|---|---|
| **NO** | Complete answer, reached free. Most of what a leaderboard shows is the maximum of thousands of random walks. Record it and stop. |
| **WEAK SIGNAL** | No individual wallet clears the bar but the top decile persists. Worth a larger scan — raise `--markets`. |
| **MAYBE** | Check bias attribution in the report. If the edge sits in extreme price bands it is favourite-longshot bias: **build the rule directly**, don't copy anyone. |
| **MAYBE**, not bias | Now the geo problem is worth solving. Reformulate around a US-legal venue. |

Remember the gate's measured false-positive rate is ~20% of populations, not the
nominal 5%. Clearing it is necessary, not sufficient — persistence is the test
that decides.